In [0]:
%sql
SHOW TABLES IN workspace.healthcare

In [0]:
%sql

-- Display all patients
SELECT *
FROM workspace.healthcare.silver_patients;

-- Find patients older than 30
SELECT PatientID, PatientName, Age, City
FROM workspace.healthcare.silver_patients
WHERE Age > 30
ORDER BY Age DESC;

In [0]:
%sql
-- Count patients by city
SELECT
    City,
    COUNT(*) AS TotalPatients,
    ROUND(AVG(Age), 2) AS AverageAge
FROM workspace.healthcare.silver_patients
GROUP BY City
ORDER BY TotalPatients DESC;

-- Count appointments by status
SELECT
    Status,
    COUNT(*) AS TotalAppointments
FROM workspace.healthcare.silver_appointments
GROUP BY Status;

In [0]:
%sql

-- Patient and doctor appointment details
SELECT
    a.AppointmentID,
    p.PatientName,
    d.DoctorName,
    d.Department,
    a.AppointmentDate,
    a.Status
FROM workspace.healthcare.silver_appointments a
INNER JOIN workspace.healthcare.silver_patients p
    ON a.PatientID = p.PatientID
INNER JOIN workspace.healthcare.silver_doctors d
    ON a.DoctorID = d.DoctorID
ORDER BY a.AppointmentDate;

-- Show all patients, including those without appointments
SELECT
    p.PatientID,
    p.PatientName,
    a.AppointmentID,
    a.Status
FROM workspace.healthcare.silver_patients p
LEFT JOIN workspace.healthcare.silver_appointments a
    ON p.PatientID = a.PatientID
ORDER BY p.PatientID;

In [0]:
%sql

-- Classify patient age groups
SELECT
    PatientID,
    PatientName,
    Age,
    CASE
        WHEN Age < 18 THEN 'Child'
        WHEN Age < 60 THEN 'Adult'
        ELSE 'Senior'
    END AS AgeCategory
FROM workspace.healthcare.silver_patients;

-- Classify billing records
SELECT
    BillingID,
    AppointmentID,
    Amount,
    PaymentStatus,
    CASE
        WHEN PaymentStatus = 'Paid' THEN 'Payment Completed'
        WHEN PaymentStatus = 'Pending' THEN 'Payment Follow-up Required'
        WHEN PaymentStatus = 'Cancelled' THEN 'Cancelled Bill'
        ELSE 'Unknown'
    END AS BillingCategory
FROM workspace.healthcare.silver_billing;

In [0]:
%sql

-- Find departments with more than one appointment

WITH DepartmentSummary AS (
    SELECT
        d.Department,
        COUNT(a.AppointmentID) AS TotalAppointments
    FROM workspace.healthcare.silver_doctors d
    INNER JOIN workspace.healthcare.silver_appointments a
        ON d.DoctorID = a.DoctorID
    GROUP BY d.Department
)

SELECT *
FROM DepartmentSummary
WHERE TotalAppointments > 1
ORDER BY TotalAppointments DESC;

In [0]:
%sql

-- Find patients who have at least one appointment

SELECT
    PatientID,
    PatientName,
    City
FROM workspace.healthcare.silver_patients
WHERE PatientID IN (
    SELECT DISTINCT PatientID
    FROM workspace.healthcare.silver_appointments
);

In [0]:
%sql

-- Rank doctors based on total appointments

WITH DoctorSummary AS (
    SELECT
        d.DoctorID,
        d.DoctorName,
        d.Department,
        COUNT(a.AppointmentID) AS TotalAppointments
    FROM workspace.healthcare.silver_doctors d
    LEFT JOIN workspace.healthcare.silver_appointments a
        ON d.DoctorID = a.DoctorID
    GROUP BY
        d.DoctorID,
        d.DoctorName,
        d.Department
)

SELECT
    DoctorID,
    DoctorName,
    Department,
    TotalAppointments,

    ROW_NUMBER() OVER (
        ORDER BY TotalAppointments DESC
    ) AS RowNumber,

    RANK() OVER (
        ORDER BY TotalAppointments DESC
    ) AS DoctorRank,

    DENSE_RANK() OVER (
        ORDER BY TotalAppointments DESC
    ) AS DenseDoctorRank

FROM DoctorSummary;

In [0]:
%sql

-- Revenue by payment status

SELECT
    PaymentStatus,
    COUNT(BillingID) AS TotalBills,
    SUM(Amount) AS TotalAmount,
    ROUND(AVG(Amount), 2) AS AverageAmount
FROM workspace.healthcare.silver_billing
GROUP BY PaymentStatus
ORDER BY TotalAmount DESC;

-- Total paid revenue and pending amount

SELECT
    SUM(
        CASE
            WHEN PaymentStatus = 'Paid' THEN Amount
            ELSE 0
        END
    ) AS TotalPaidRevenue,

    SUM(
        CASE
            WHEN PaymentStatus = 'Pending' THEN Amount
            ELSE 0
        END
    ) AS TotalPendingAmount

FROM workspace.healthcare.silver_billing;

In [0]:
%sql

CREATE OR REPLACE VIEW workspace.healthcare.vw_patient_appointments AS

SELECT
    a.AppointmentID,
    p.PatientID,
    p.PatientName,
    p.Age,
    p.City,
    d.DoctorName,
    d.Department,
    a.AppointmentDate,
    a.Status
FROM workspace.healthcare.silver_appointments a
INNER JOIN workspace.healthcare.silver_patients p
    ON a.PatientID = p.PatientID
INNER JOIN workspace.healthcare.silver_doctors d
    ON a.DoctorID = d.DoctorID;

In [0]:
%sql

SELECT *
FROM workspace.healthcare.vw_patient_appointments;

In [0]:
%sql

SELECT
    d.Department,
    COUNT(DISTINCT a.AppointmentID) AS TotalAppointments,

    SUM(
        CASE
            WHEN a.Status = 'Completed' THEN 1
            ELSE 0
        END
    ) AS CompletedAppointments,

    SUM(
        CASE
            WHEN b.PaymentStatus = 'Paid' THEN b.Amount
            ELSE 0
        END
    ) AS PaidRevenue,

    SUM(
        CASE
            WHEN b.PaymentStatus = 'Pending' THEN b.Amount
            ELSE 0
        END
    ) AS PendingRevenue

FROM workspace.healthcare.silver_appointments a

INNER JOIN workspace.healthcare.silver_doctors d
    ON a.DoctorID = d.DoctorID

LEFT JOIN workspace.healthcare.silver_billing b
    ON a.AppointmentID = b.AppointmentID

GROUP BY d.Department
ORDER BY TotalAppointments DESC;